In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append(
    "./modules/python-utils:./modules/ai-utils"
)
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import librosa
import numpy as np
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/"
DESTINATION = "/workspaces/dev/output/LibriSpeechASRcorpus/sclient/rt_whisper/test-other/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
HYPERPARAMETER_PATH = "./hyperparameters/sclite.yml"

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter_path=HYPERPARAMETER_PATH)

In [ ]:
src = Path(SOURCE)
dest = Path(DESTINATION)
dest.mkdir(parents=True, exist_ok=True)

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

    completed = []
    param = Param()
    # start_time = time.perf_counter()
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        result:Result = token_streamer.process(param)
        completed.extend(result.completed)
        param.update(result)
    completed.extend(result.candidate)
    # end_time = time.perf_counter()
    # print(f"Processed {flac.stem} in {end_time - start_time:.2f} seconds")

    return TRNFormat(
        id = flac.stem,
        text = " ".join([s.text for s in completed])
    )

In [ ]:
%%time
make_all_ref_and_hyp(src, dest, transcriber)
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.ref.trn"))),
    dest / "concat.ref.trn"
)
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.hyp.trn"))),
    dest / "concat.hyp.trn"
)
# CPU times: user 5h 39min 24s, sys: 11min 32s, total: 5h 50min 57s
# Wall time: 1h 13min 21s

In [ ]:
output = sclite_trn_run(
    dest / "concat.ref.trn",
    dest / "concat.hyp.trn",
)

In [ ]:
parse_sclite_summary(output)

# {'num_sentences': 2939,
#  'num_words': 52343,
#  'correct_percent': 83.6,
#  'substitution_percent': 14.0,
#  'deletion_percent': 2.5,
#  'insertion_percent': 0.9,
#  'wer_percent': 17.4,
#  'sentence_error_percent': 94.9}